In [1]:
# Install dependencies
!pip install -q timm pytorch-msssim lpips scikit-image thop gdown transformers accelerate

import os
import shutil
import subprocess
import json

# --- 1. MOUNTED ARTIFACT PATHS (UPDATE THESE) ---
MOUNTED_SATEHAZE_WEIGHTS_BASE = '/kaggle/input/notebooks/markosumire/nonvlm/weights/SateHaze1k_Baseline/model_best.pth'
MOUNTED_SATEHAZE_WEIGHTS_VLM  = '/kaggle/input/notebooks/markosumire/vlmtraining/weights/SateHaze1k_VLM/model_best.pth'

# Update this to point to the `captions` folder from your RICE captioning run
MOUNTED_RICE_CAPTIONS_BASE    = '/kaggle/input/notebooks/markosumire/captioningrice/captions'

# --- 2. CLONE REPOSITORY ---
!git clone https://github.com/sumire25/RSHazeNet.git /kaggle/working/RSHazeNet
%cd /kaggle/working/RSHazeNet

# --- 3. PATH RESOLUTION FOR RICE DATASETS ---
RICE1_CLOUD = '/kaggle/input/datasets/shubhank001/rice-remote-sensing-images-for-cloud-removal/RICE/RICE1/cloud'
RICE1_CLEAR = '/kaggle/input/datasets/shubhank001/rice-remote-sensing-images-for-cloud-removal/RICE/RICE1/clear'
RICE2_CLOUD = '/kaggle/input/datasets/shubhank001/rice-remote-sensing-images-for-cloud-removal/RICE/RICE2/cloud'
RICE2_CLEAR = '/kaggle/input/datasets/shubhank001/rice-remote-sensing-images-for-cloud-removal/RICE/RICE2/clear'

# Fallbacks for Kaggle's dynamic mounting structure
if not os.path.exists(RICE1_CLOUD):
    RICE1_CLOUD = '/kaggle/input/rice-remote-sensing-images-for-cloud-removal/RICE/RICE1/cloud'
    RICE1_CLEAR = '/kaggle/input/rice-remote-sensing-images-for-cloud-removal/RICE/RICE1/clear'
if not os.path.exists(RICE2_CLOUD):
    RICE2_CLOUD = '/kaggle/input/rice-remote-sensing-images-for-cloud-removal/RICE/RICE2/cloud'
    RICE2_CLEAR = '/kaggle/input/rice-remote-sensing-images-for-cloud-removal/RICE/RICE2/clear'

# --- 4. RICE DATASET AGGREGATOR ---
def prepare_rice_layout(dataset_name, cloud_dir, clear_dir, unified_base):
    if os.path.exists(unified_base): shutil.rmtree(unified_base)
    
    hazy_dir = os.path.join(unified_base, 'test', 'hazy')
    gt_dir = os.path.join(unified_base, 'test', 'GT')
    os.makedirs(hazy_dir, exist_ok=True)
    os.makedirs(gt_dir, exist_ok=True)
    
    for fname in os.listdir(cloud_dir):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')): continue
        
        # RICE is 512x512, perfectly matching SateHaze1k, so we symlink directly
        os.symlink(os.path.join(cloud_dir, fname), os.path.join(hazy_dir, fname))
        os.symlink(os.path.join(clear_dir, fname), os.path.join(gt_dir, fname))
        
    # Retrieve matching captions using the directory key generated by caption.py
    cap_key = cloud_dir.replace('/', '_').strip('_')
    cap_path = os.path.join(MOUNTED_RICE_CAPTIONS_BASE, cap_key, 'captions.json')
    
    if os.path.exists(cap_path):
        with open(cap_path, 'r') as f:
            captions = json.load(f)
        with open(os.path.join(hazy_dir, 'captions.json'), 'w') as f:
            json.dump(captions, f, indent=4)
    else:
        print(f"[WARNING] No captions found for {dataset_name} at {cap_path}")

UNIFIED_RICE1 = '/kaggle/working/Unified_RICE1'
UNIFIED_RICE2 = '/kaggle/working/Unified_RICE2'

print("Preparing RICE1 Dataset Layout...")
prepare_rice_layout("RICE1", RICE1_CLOUD, RICE1_CLEAR, UNIFIED_RICE1)

print("Preparing RICE2 Dataset Layout...")
prepare_rice_layout("RICE2", RICE2_CLOUD, RICE2_CLEAR, UNIFIED_RICE2)

# --- 5. TEST ORCHESTRATOR ---
def run_cross_domain_test(model_name, target_dataset, unified_base, weights_path, use_vlm=False):
    mode_str = "VLM" if use_vlm else "Baseline"
    print(f"\n{'='*50}\nTesting Cross-Domain ({mode_str}): {model_name} on {target_dataset}\n{'='*50}")
    
    result_dir = f'/kaggle/working/results/CrossDomain_{model_name}_to_{target_dataset}_{mode_str}'
    os.makedirs(result_dir, exist_ok=True)
    
    # 1. Inference Call
    test_cmd = [
        'python', 'infer.py',
        '--test_input', f'{unified_base}/test/hazy',
        '--test_target', f'{unified_base}/test/GT',
        '--weights', weights_path,
        '--result_path', result_dir
    ]
    if use_vlm:
        test_cmd.append('--caption')
        
    print(f"-> Running Inference ({mode_str})...")
    subprocess.run(test_cmd, check=True)
    
    # 2. Metric Evaluation & Preservation
    eval_cmd = [
        'python', 'evaluate.py',
        '--train_folder', result_dir,
        '--target_folder', f'{unified_base}/test/GT'
    ]
    print("-> Computing Evaluation Metrics...")
    
    metrics_file = os.path.join(result_dir, 'metrics.txt')
    with open(metrics_file, 'w') as f:
        subprocess.run(eval_cmd, check=True, stdout=f, text=True)
        
    with open(metrics_file, 'r') as f:
        print(f.read())

# --- 6. EXECUTE PIPELINE ---

# RICE1 Evaluation
run_cross_domain_test("SateHaze1k", "RICE1", UNIFIED_RICE1, MOUNTED_SATEHAZE_WEIGHTS_BASE, use_vlm=False)
run_cross_domain_test("SateHaze1k", "RICE1", UNIFIED_RICE1, MOUNTED_SATEHAZE_WEIGHTS_VLM, use_vlm=True)

# RICE2 Evaluation
run_cross_domain_test("SateHaze1k", "RICE2", UNIFIED_RICE2, MOUNTED_SATEHAZE_WEIGHTS_BASE, use_vlm=False)
run_cross_domain_test("SateHaze1k", "RICE2", UNIFIED_RICE2, MOUNTED_SATEHAZE_WEIGHTS_VLM, use_vlm=True)

print("\nRICE Cross-Domain Testing Complete! Outputs and metrics are available in /kaggle/working/results/")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.7 MB/s eta 0:00:00
Cloning into '/kaggle/working/RSHazeNet'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 126 (delta 71), reused 85 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 2.88 MiB | 13.40 MiB/s, done.
Resolving deltas: 100% (71/71), done.
/kaggle/working/RSHazeNet
Preparing Cross-Domain Dataset (Scaling hazy images to 512x512)...

Testing Cross-Domain (Baseline): SateHaze1k on RRSHID
-> Running Inference (Baseline)...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Inference: 100%|██████████| 304/304 [00:28<00:00, 10.82it/s]
-> Computing Evaluation Metrics...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 233M/233M [00:01<00:00, 172MB/s]
100%|██████████| 104M/104M [00:00<00:00, 167MB/s] 


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
SAM: 8.533
ERGAS: 130.517
UIQI: 0.368
QNR: 0.143
BRISQUE: 1.478
NIQE: 3.089
Histogram: 0.164
Spectral Curve Deviation: 0.508
PSNR: 12.362
SSIM: 0.368
CIEDE2000: 24.473
LPIPS: 0.592
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth
FID: 207.950


Testing Cross-Domain (VLM): SateHaze1k on RRSHID
-> Running Inference (VLM)...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2355.92it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Caption] Loaded 304 test captions.
Inference: 100%|██████████| 304/304 [00:32<00:00,  9.29it/s]
-> Computing Evaluation Metrics...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
SAM: 14.214
ERGAS: 167.530
UIQI: 0.365
QNR: 0.073
BRISQUE: 1.483
NIQE: 2.258
Histogram: 0.124
Spectral Curve Deviation: 0.455
PSNR: 10.151
SSIM: 0.365
CIEDE2000: 33.115
LPIPS: 0.585
FID: 154.664


Cross-Domain Testing Complete! Outputs and metrics are available in /kaggle/working/results/
